# Step 05 - Editor Agent

Goal: generate clean final article JSON for publishing.

## What's New in This Step

- Step 04 ended with Markdown from the writer.
- This step adds an `editor_agent` that converts Markdown into publish-ready JSON.
- Introduction to guardrails:
  - Format guardrail: return strict JSON only.
  - Shape guardrail: enforce required keys.
  - Quality guardrail: keep tags clean, short, and bounded.

### Setup and Define Helper Functions

In [ ]:
import json
import os
from dotenv import load_dotenv
from crewai import LLM, Agent, Crew, Task
from crewai_tools import SerperDevTool

load_dotenv()

TOPIC = "Platform Engineering Best Practices"

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
serper_api_key = os.getenv("SERPER_API_KEY")

if not openrouter_api_key:
    raise ValueError("Missing OPENROUTER_API_KEY")

if not serper_api_key:
    raise ValueError("Missing SERPER_API_KEY")


def extract_json_object(text):
    # Format guardrail recovery: strip fences and parse first valid JSON object.
    cleaned = text.strip().replace("```json", "").replace("```", "").strip()
    decoder = json.JSONDecoder()

    for i, char in enumerate(cleaned):
        if char != "{":
            continue

        try:
            obj, _ = decoder.raw_decode(cleaned[i:])
            return obj
        except json.JSONDecodeError:
            continue

    raise ValueError("Could not extract valid JSON object from model output.")

### Create LLM and Agents

In [ ]:
# LLM: shared model used by all agents in this pipeline.
llm = LLM(
    model="openai/gpt-4o",
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,
)

# Built-in search tool for fresh references.
search_tool = SerperDevTool()

# Agent 1: gathers source-backed notes.
research_agent = Agent(
    role="Research Analyst",
    goal="Find latest updates on {topic}",
    backstory="You give short, factual, source-backed summaries.",
    llm=llm,
    tools=[search_tool],
    verbose=False,
)

# Agent 2: creates Markdown draft from research.
writer_agent = Agent(
    role="Content Writer",
    goal="Write a simple blog post on {topic}",
    backstory="You write clear and beginner-friendly content.",
    llm=llm,
    verbose=False,
)

# Agent 3: acts as validator/formatter to produce strict JSON for downstream publish.
editor_agent = Agent(
    role="Editor",
    goal="Return final output in JSON",
    backstory="You return only valid JSON with title, tags, and content.",
    llm=llm,
    verbose=False,
)

### Define Tasks with Context

In [ ]:
# Task 1: research with dated sources.
research_task = Task(
    description="Research {topic} from 2026 onward with source links.",
    expected_output="A short research summary with sources.",
    agent=research_agent,
)

# Task 2: convert research into Markdown content.
writing_task = Task(
    description="Write a clear blog post from the research notes.",
    expected_output="A markdown blog post.",
    agent=writer_agent,
    context=[research_task],
)

# Task 3 (guardrails):
# - Format guardrail: valid JSON only
# - Shape guardrail: required keys title/tags/content
# - Quality guardrail: up to 4 short tags
editing_task = Task(
    description=(
        "Return only valid JSON with keys: title, tags, content. "
        "tags must be a list of up to 4 short and relevant tags."
    ),
    expected_output='{"title": "...", "tags": ["ai"], "content": "markdown"}',
    agent=editor_agent,
    context=[writing_task],
)


### Create Crew with Agents and Tasks

In [ ]:
crew = Crew(
    agents=[research_agent, writer_agent, editor_agent],
    tasks=[research_task, writing_task, editing_task],
    verbose=False,
)

### Kickoff the Crew with the topic input and extract the final article JSON, applying guardrail normalization.

In [ ]:
result = crew.kickoff(inputs={"topic": TOPIC})
result_text = getattr(result, "raw", str(result))

article_data = extract_json_object(result_text)

# Quality guardrail normalization: convert tags to a clean max-4 list.
raw_tags = article_data.get("tags", [])

if isinstance(raw_tags, str):
    raw_tags = [raw_tags]

article_data["tags"] = [
    str(tag).lower().replace(" ", "")
    for tag in raw_tags
][:4]

if not article_data["tags"]:
    article_data["tags"] = ["ai", "agents"]

print(json.dumps(article_data, indent=2))

### Save final article JSON to file for downstream publishing.

In [ ]:
with open("article_output.json", "w", encoding="utf-8") as file:
    json.dump(article_data, file, indent=2, ensure_ascii=False)

print("Saved final article JSON to article_output.json")

### Guardrail demo: what happens when the editor returns junk?

The cells above show the *happy path*. To prove the guardrail is real, feed
some realistic bad model output into `extract_json_object` and see it
recover or fail loudly.

In [ ]:
# Case 1: model wraps JSON in a code fence and adds a chatty preamble.
fenced_output = """
Sure! Here is the article in the requested format:

```json
{"title": "Platform Engineering in 2026", "tags": ["platform", "devops"], "content": "# Intro\n..."}
```

Let me know if you need changes.
"""

print("Case 1 (fenced + preamble):")
print(json.dumps(extract_json_object(fenced_output), indent=2))

# Case 2: model returns prose with no JSON object at all -> guardrail must fail.
prose_only = "I think the title should probably be about platform engineering."

print("\nCase 2 (no JSON):")
try:
    extract_json_object(prose_only)
except ValueError as err:
    print(f"Guardrail rejected output: {err}")

### Recap
- LLM did: generate content and transform it into a structured output.
- Agents did: split writing from editing so validation is a separate responsibility.
- Task enforced: explicit format, shape, and quality guardrails before publish.